In [ ]:
![ -d /kaggle/working/codapath/.git ] || git clone https://github.com/CryAndRRich/codapath.git /kaggle/working/codapath

In [ ]:
%cd /kaggle/working/codapath
CODAPATH = "/kaggle/working/codapath"

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

In [ ]:
# pathmnist | histoset | skintissue
DATASET = "histoset"

# random | coreset | typiclust | activeft | badge | entropy | margin
# codapath | scalpel | nucleus_al | uncertainty_herding | tcm | dropquery | refine
SAMPLER_NAME = "codapath"

SEED = 42

# Optional explicit output prefix. nucleus_al derives one from source+uncertainty
# when this is None, so its variants do not overwrite each other.
RUN_NAME = None

# Mounted or local pre-extracted caches.
FEATURE_DIR = "features"
NUCLEUS_FEATURE_DIR = "nucleus_features"

In [ ]:
import os
from huggingface_hub import snapshot_download

# DINOv2 is public: no placeholder login/token is required.
# Enable Kaggle Internet, or set the model path to a mounted local snapshot.
print("Downloading facebook/dinov2-base...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if CODAPATH not in sys.path:
    sys.path.append(CODAPATH)

In [ ]:
import yaml
import torch

from run import main

In [ ]:
PATHMNIST_PATH  = "/kaggle/input/datasets/cryandrrich/nckh2026/pathmnist_224.npz"
HISTOSET_PATH   = "/kaggle/input/datasets/cryandrrich/nckh2026/HistoSet-5x14/HistoSet-5x14"
SKINTISSUE_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/SkinTissue/SkinTissue/tiles"

DATA_DICT = {
    "pathmnist":  PATHMNIST_PATH,
    "histoset":   HISTOSET_PATH,
    "skintissue": SKINTISSUE_PATH,
}

In [ ]:
from pathlib import Path

CONFIG_PATH = "config/config.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

training_cfg = config.get("training", {})
dataset_info = config["datasets"][DATASET]
sampler_cfg  = config.get("samplers", {}).get(SAMPLER_NAME, {})

# Override sampler hyperparams from the notebook (no repo/config edit needed).
# e.g. nucleus variant: {"cell_source": "crop_dino", "uncertainty_mode": "cell_margin"}
SAMPLER_OVERRIDES = {}
sampler_cfg = {**sampler_cfg, **SAMPLER_OVERRIDES}
if SAMPLER_NAME == "nucleus_al":
    nucleus_manifest = (
        Path(NUCLEUS_FEATURE_DIR) / f"{DATASET}_seed{SEED}" / "manifest.json"
    )
    assert nucleus_manifest.is_file(), (
        f"Missing nucleus cache: {nucleus_manifest}. Mount the extraction notebook output."
    )
print(f"[{SAMPLER_NAME}] sampler_cfg =", sampler_cfg)

In [ ]:
main(
    data_path=DATA_DICT[DATASET],
    sampler_name=SAMPLER_NAME,
    num_classes=dataset_info["num_classes"],
    cumulative_budget=config["cumulative_budget"],
    data_descriptions=dataset_info["descriptions"],
    prompt_templates=config["prompt_templates"],
    sampler_cfg=sampler_cfg,
    probe_epochs=training_cfg["probe_epochs"],
    probe_lr=training_cfg["probe_lr"],
    device=torch.device(config["device"]),
    random_seed=SEED,
    save_dir=f"checkpoints/{DATASET}",
    verbose=True,
    model_cfg=config.get("models", {}),
    feature_cache_dir=FEATURE_DIR,
    nucleus_cache_dir=NUCLEUS_FEATURE_DIR,
    run_name=RUN_NAME,
)